# **07. Model Evaluation**

### **Introduction**

Multiple baseline machine learning models were trained and evaluated using the ROC-AUC score. The best-performing model was selected for hyperparameter tuning using GridSearchCV. The optimized model was then retrained with the best parameters and evaluated using standard classification metrics, including accuracy, precision, recall, F1-score, and ROC-AUC.


In [20]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sidetable
import sklearn
import feature_engine
import scipy
from scipy import stats
import time
from pathlib import Path
import pickle
import joblib
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    precision_score,
    recall_score,
    f1_score
)



In [21]:
# Display Settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
from IPython.display import display, Markdown

def display_md(string):
    display(Markdown(string))
    

## **1. Loading Preprocessed Model Data**

In [22]:
# define path for best model
parent_path = Path.cwd().parent
best_model_path = parent_path.joinpath('models', 'best_model.pkl')


In [23]:
display_md("**Loading Best Model Data....**")
with open(best_model_path, 'rb') as file:
    best_model_data = joblib.load(file)
    

**Loading Best Model Data....**

In [24]:
best_model_data.keys()

dict_keys(['model', 'best_pipeline', 'selected_features', 'full_preprocessed_train', 'y_train', 'full_preprocessed_test', 'y_test'])

In [25]:
# Load the best model and its components
best_model = best_model_data['model']
best_model_pipeline = best_model_data['best_pipeline']
selected_features = best_model_data['selected_features']
X_train = best_model_data['full_preprocessed_train']
X_test = best_model_data['full_preprocessed_test']
y_train = best_model_data['y_train']
y_test = best_model_data['y_test']
X_train_new = X_train[selected_features]
X_test_new = X_test[selected_features]


## **2. Hyperparameter Tuning for XGBoost Classifier**

In [26]:
# store tuning results in list
tuning_results = []

# Initialize the XGBClassifier
xgb_model_pipeline = best_model_pipeline

# Define parameters grid
parameters_dict = {
    'classifier__min_child_weight': [1, 5, 10],
    'classifier__gamma': [0.5, 1, 1.5, 2, 5],
    'classifier__subsample': [0.6, 0.8, 1.0],       # fraction of samples to be used for fitting the individual base learners
    'classifier__colsample_bytree': [0.6, 0.8, 1.0], # fraction of features to be used for fitting the individual base learners
    'classifier__max_depth': [3, 4, 5]
}

# Set up stratified k-fold cross-validation
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Set up GridSearchCV
grid_search = GridSearchCV(
    estimator=xgb_model_pipeline,
    param_grid=parameters_dict,
    scoring='roc_auc',
    n_jobs=-1,
    cv=kfold,
    verbose=1
)

# Measuring training time
start_time = time.time()
# Fit the model
grid_search.fit(X_train_new, y_train)
end_time = time.time()
training_time = end_time - start_time

# Get the best parameters and best score
best_params = grid_search.best_params_
best_score = grid_search.best_score_

print(f"Best Parameters: {best_params}")
print(f"Best ROC AUC Score: {best_score}")
print(f"Training Time: {training_time} seconds")

tuning_results.append({
    'model': best_model,
    'best_params': best_params,
    'best_roc_auc_score': best_score,
    'training_time': training_time
})

# Save the tuning results as dataframe
tuning_results_df = pd.DataFrame(tuning_results)


Fitting 5 folds for each of 405 candidates, totalling 2025 fits


c:\Users\Admin\anaconda3\envs\test_titanic\Lib\site-packages\xgboost\training.py:200: UserWarning: [21:14:38] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Best Parameters: {'classifier__colsample_bytree': 0.8, 'classifier__gamma': 2, 'classifier__max_depth': 4, 'classifier__min_child_weight': 10, 'classifier__subsample': 1.0}
Best ROC AUC Score: 0.9295482126066303
Training Time: 796.2121765613556 seconds
